# 2_Ingesta_Bronze

## Objetivo
Ingerir los cinco archivos CSV desde la capa **Raw** de Azure Data Lake Storage Gen2 hacia la capa **Bronze**, almacenándolos en formato Delta y registrando metadatos técnicos de ingesta.

Se utiliza Managed Identity mediante las External Locations configuradas en Unity Catalog. La lectura inicial permite observar la inferencia de tipos de Spark y la segunda lectura aplica un esquema explícito definido para cada dataset.

## 1. Configuración inicial

In [ ]:
dbutils.widgets.removeAll()

In [ ]:
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DateType, TimestampType, DecimalType
)

## 2. Parámetros de ejecución

In [ ]:
dbutils.widgets.text("containerRaw", "raw")
dbutils.widgets.text("containerDataLake", "datalake")
dbutils.widgets.text("storageName", "stproyectofinalehs")
dbutils.widgets.text("catalogo", "dbw_proyecto_final_ehs")

In [ ]:
containerRaw = dbutils.widgets.get("containerRaw")
containerDataLake = dbutils.widgets.get("containerDataLake")
storageName = dbutils.widgets.get("storageName")
catalogo = dbutils.widgets.get("catalogo")

esquema = "bronze"

ruta_raw = f"abfss://{containerRaw}@{storageName}.dfs.core.windows.net/"
ruta_bronze = f"abfss://{containerDataLake}@{storageName}.dfs.core.windows.net/bronze/"

print("Catálogo:", catalogo)
print("Esquema:", esquema)
print("Ruta Raw:", ruta_raw)
print("Ruta Bronze:", ruta_bronze)

## 3. Ingesta de almacenes

In [ ]:
# Lectura inicial para observar los tipos inferidos por Spark
df_almacenes_inferido = spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
    .csv(f"{ruta_raw}almacenes.csv")

df_almacenes_inferido.printSchema()

# Definición manual del esquema
almacenes_schema = StructType([
    StructField("almacen_id", IntegerType(), True),
    StructField("nombre_almacen", StringType(), True),
    StructField("ciudad", StringType(), True),
    StructField("region", StringType(), True),
    StructField("es_activo", IntegerType(), True)
])

# Lectura aplicando el esquema definido
df_almacenes = spark.read\
    .option("header", True)\
    .schema(almacenes_schema)\
    .csv(f"{ruta_raw}almacenes.csv")

df_almacenes = df_almacenes\
    .withColumn("_source_file", lit("almacenes.csv"))\
    .withColumn("_ingestion_timestamp", current_timestamp())

df_almacenes.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .option("path", f"{ruta_bronze}almacenes/")\
    .saveAsTable(f"{catalogo}.{esquema}.almacenes")

print("Almacenes:", df_almacenes.count())

## 4. Ingesta de clientes

In [ ]:
# Lectura inicial para observar los tipos inferidos por Spark
df_clientes_inferido = spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
    .csv(f"{ruta_raw}clientes.csv")

df_clientes_inferido.printSchema()

# Definición manual del esquema
# documento se conserva como StringType por ser un identificador, no una cantidad.
clientes_schema = StructType([
    StructField("cliente_id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("tipo_documento", StringType(), True),
    StructField("documento", StringType(), True),
    StructField("ciudad", StringType(), True),
    StructField("fecha_registro", DateType(), True),
    StructField("tipo_cliente", StringType(), True)
])

# Lectura aplicando el esquema definido
df_clientes = spark.read\
    .option("header", True)\
    .option("dateFormat", "yyyy-MM-dd")\
    .schema(clientes_schema)\
    .csv(f"{ruta_raw}clientes.csv")

df_clientes = df_clientes\
    .withColumn("_source_file", lit("clientes.csv"))\
    .withColumn("_ingestion_timestamp", current_timestamp())

df_clientes.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .option("path", f"{ruta_bronze}clientes/")\
    .saveAsTable(f"{catalogo}.{esquema}.clientes")

print("Clientes:", df_clientes.count())

## 5. Ingesta de inventario

In [ ]:
# Lectura inicial para observar los tipos inferidos por Spark
df_inventario_inferido = spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
    .csv(f"{ruta_raw}inventario.csv")

df_inventario_inferido.printSchema()

# Definición manual del esquema
inventario_schema = StructType([
    StructField("fecha_corte", DateType(), True),
    StructField("producto_id", IntegerType(), True),
    StructField("almacen_id", IntegerType(), True),
    StructField("stock_disponible", IntegerType(), True),
    StructField("stock_minimo", IntegerType(), True)
])

# Lectura aplicando el esquema definido
df_inventario = spark.read\
    .option("header", True)\
    .option("dateFormat", "yyyy-MM-dd")\
    .schema(inventario_schema)\
    .csv(f"{ruta_raw}inventario.csv")

df_inventario = df_inventario\
    .withColumn("_source_file", lit("inventario.csv"))\
    .withColumn("_ingestion_timestamp", current_timestamp())

df_inventario.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .option("path", f"{ruta_bronze}inventario/")\
    .saveAsTable(f"{catalogo}.{esquema}.inventario")

print("Inventario:", df_inventario.count())

## 6. Ingesta de productos

In [ ]:
# Lectura inicial para observar los tipos inferidos por Spark
df_productos_inferido = spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
    .csv(f"{ruta_raw}productos.csv")

df_productos_inferido.printSchema()

# Definición manual del esquema
productos_schema = StructType([
    StructField("producto_id", IntegerType(), True),
    StructField("nombre_producto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("marca", StringType(), True),
    StructField("costo_unitario", DecimalType(12, 2), True),
    StructField("precio_venta", DecimalType(12, 2), True),
    StructField("es_activo", IntegerType(), True)
])

# Lectura aplicando el esquema definido
df_productos = spark.read\
    .option("header", True)\
    .schema(productos_schema)\
    .csv(f"{ruta_raw}productos.csv")

df_productos = df_productos\
    .withColumn("_source_file", lit("productos.csv"))\
    .withColumn("_ingestion_timestamp", current_timestamp())

df_productos.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .option("path", f"{ruta_bronze}productos/")\
    .saveAsTable(f"{catalogo}.{esquema}.productos")

print("Productos:", df_productos.count())

## 7. Ingesta de ventas

In [ ]:
# Lectura inicial para observar los tipos inferidos por Spark
df_ventas_inferido = spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
    .csv(f"{ruta_raw}ventas.csv")

df_ventas_inferido.printSchema()

# Definición manual del esquema
ventas_schema = StructType([
    StructField("venta_id", IntegerType(), True),
    StructField("fecha_venta", TimestampType(), True),
    StructField("cliente_id", IntegerType(), True),
    StructField("producto_id", IntegerType(), True),
    StructField("almacen_id", IntegerType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio_unitario", DecimalType(12, 2), True),
    StructField("descuento_pct", DecimalType(5, 2), True),
    StructField("metodo_pago", StringType(), True)
])

# Lectura aplicando el esquema definido
df_ventas = spark.read\
    .option("header", True)\
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")\
    .schema(ventas_schema)\
    .csv(f"{ruta_raw}ventas.csv")

df_ventas = df_ventas\
    .withColumn("_source_file", lit("ventas.csv"))\
    .withColumn("_ingestion_timestamp", current_timestamp())

df_ventas.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .option("path", f"{ruta_bronze}ventas/")\
    .saveAsTable(f"{catalogo}.{esquema}.ventas")

print("Ventas:", df_ventas.count())

## 8. Validación de la ingesta

In [ ]:
tablas_bronze = ["almacenes", "clientes", "inventario", "productos", "ventas"]

total_registros = 0

for tabla in tablas_bronze:
    nombre_tabla = f"{catalogo}.{esquema}.{tabla}"
    df_validacion = spark.table(nombre_tabla)
    cantidad = df_validacion.count()
    total_registros += cantidad

    print(f"{tabla}: {cantidad} registros")
    df_validacion.printSchema()

print("Total de registros en Bronze:", total_registros)

## 9. Resultado esperado

| Tabla | Registros esperados |
|---|---:|
| almacenes | 20 |
| clientes | 10,012 |
| inventario | 10,000 |
| productos | 1,000 |
| ventas | 100,020 |
| **Total** | **121,052** |

La capa Bronze conserva los registros de origen, incluidos los valores nulos y las incidencias detectadas durante el perfilado. La depuración y las reglas de calidad se implementarán en Silver.

Cada tabla incluye las columnas técnicas `_source_file` y `_ingestion_timestamp`.